# SIR versus the corrected sampler

Two comparisons of the two ways to obtain **unweighted** paths from the projected measure:
**SIR** (draw from $P_\theta$, weight by $L^*$, resample with replacement) and the **corrected sampler**
(the frozen reverse chain with the $\varepsilon$-hook $-\sqrt{1-\bar\alpha}\,\nabla\log h_\psi$).

**A — headline configuration** (C3, baseline prior, ESS $\approx$ 93 %): standard error per wall-clock
minute at matched compute, for the three exotics, plus the effective diversity of the resampled set
(unique draws retained).

**B — crossover sweep along the KL-budget curve at C3.** The curve spans ESS 93 % $\to$ 32 % by tilting
the projected measure toward and away from a payoff. **This is a mechanism test, not a pricing
configuration:** the tilts are payoff-directed, so the measures on this curve are not ones anybody would
price with. The point is to locate the crossover empirically rather than assume it.

### Pre-registered expectations

Let $p$ be the correction premium (corrected draw / uncorrected draw, measured at 1.71 in the
amortization artifact) and $e$ the ESS fraction. At matched compute:

| comparator | sampler wins when | crossover |
|---|---|---|
| weighted IS (weights kept) | $p < 1/e$ | $e = 1/p = 58.6\,\%$ |
| **SIR** (resampled to unweighted) | $p < 1 + 1/e$ | $e = 1/(p-1) = 141\,\%$ |

SIR's estimator carries the weighting variance **and** the resampling variance,
$\mathrm{Var} \approx \sigma_Q^2(1/e + 1)/N$, so against SIR the sampler is predicted to win at
**every** attainable ESS — about 9 % lower SE at $e = 0.93$ and 36 % at $e = 0.32$.

**Two expectations are therefore on the record and they disagree.** The stated expectation is that SIR
matches or beats the sampler at 93 % and loses below the crossover. The derivation above says the
sampler wins throughout. Whichever the run supports is reported plainly; if SIR wins at 93 % the
derivation is wrong and that is the finding.

### Cell 0 — clone, pin, fail fast if stale

In [ ]:
PINNED_COMMIT    = "__PINNED__"
NOTEBOOK_VERSION = "sir-2026.09.23a"
EXPECT_TASKC     = "taskc-2026.09.23a"

%cd /content
%rm -rf ddpm_option_pricing
!git clone -q https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch -q --all
!git checkout -q {PINNED_COMMIT}
import subprocess
HEAD = subprocess.run(["git","rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
print("pinned :", PINNED_COMMIT); print("HEAD   :", HEAD); print("notebook:", NOTEBOOK_VERSION)
assert HEAD.startswith(PINNED_COMMIT) or PINNED_COMMIT.startswith(HEAD), "checkout missed the pinned commit -- stop"
!git log --oneline -1

### Cell 1 — imports, full fp32, version guard

In [ ]:
import os, sys, json, math, time, pickle
import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, matplotlib.pyplot as plt
from dataclasses import asdict, replace
import taskc
from taskc.config import CFG, FROZEN, frozen
from taskc.data import build_training_set, reference_paths
from taskc.ptheta import make_schedule, load_checkpoint, sample_ptheta, save_draw, Schedule
from taskc.gate import run_gate, print_report, summary_dict
from taskc.dual import solve_level
from taskc.hnet import HNet, train_hnet, save_hnet
from taskc.smt import sample_smt, resample_weighted, make_eps_correction
from taskc.kl_budget import standardise, solve_beta, kl_of
from constraints import build
from config import q_params, CONSTRAINT_LEVELS, EXOTICS
import evaluation as ev

assert taskc.__version__ == EXPECT_TASKC, f"stale: taskc {taskc.__version__} != {EXPECT_TASKC}"
assert NOTEBOOK_VERSION in open("notebooks/taskc_07_sir_vs_sampler_colab.ipynb").read(), "stale notebook -- reopen from the pinned URL"
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "")
print("tf32:", torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32, "| precision:", torch.get_float32_matmul_precision())
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/ddpm_option_pricing/artifacts_sir"
else:
    DRIVE = os.environ.get("DRY_OUT", "artifacts_sir_local")
os.makedirs(DRIVE, exist_ok=True); print("writing to", DRIVE)
def close(a,b,rtol=1e-6,atol=1e-12): return math.isclose(float(a),float(b),rel_tol=rtol,abs_tol=atol)

### Cell 2 — configuration and resumable state

In [ ]:
LEVEL   = "C3"
N_EVAL  = 100_000          # paths per method in comparison A
N_POOL  = 1_000_000        # solve draw / SIR pool
R_SIR   = 20               # independent resamples, for an empirical SIR standard error
SWEEP_EXOTIC = "lookback_float_call"     # widest ESS span on the KL curve
TARGET_ESS   = [0.93, 0.80, 0.65, 0.50, 0.40, 0.32]
SKIP_DONE = True
RUN = frozen(artifact_dir=DRIVE)
RES = os.path.join(DRIVE, "sir_vs_sampler.json")
res = json.load(open(RES)) if (SKIP_DONE and os.path.exists(RES)) else dict(
    notebook=NOTEBOOK_VERSION, pinned=PINNED_COMMIT, taskc=taskc.__version__, device=DEVICE,
    device_name=(torch.cuda.get_device_name(0) if DEVICE=="cuda" else DEVICE),
    fp32=dict(tf32_matmul=torch.backends.cuda.matmul.allow_tf32, precision=torch.get_float32_matmul_precision()),
    level=LEVEL, n_eval=N_EVAL, n_pool=N_POOL, stages={}, timings={})
def save(): json.dump(res, open(RES,"w"), indent=1, default=float)
save(); print("stages done:", list(res["stages"]))

### Stage 1 — the baseline prior

The frozen $P_\theta$ is loaded from `models/frozen_ptheta/`, the checkpoint the rest of the paper uses,
so this is the headline prior rather than a refit. The acceptance gate is re-run as a device-consistency
check; it passed at $10^5$ on the machine that produced it.

In [ ]:
ts = build_training_set(RUN); sched = make_schedule(RUN, device=DEVICE)
model, std_ck, ck_cfg, ck_extra = load_checkpoint("models/frozen_ptheta/ptheta_mlp21.pt", device=DEVICE)
for f in ("m","s","S0","r","dt"): assert close(getattr(std_ck,f), getattr(ts.std,f)), f"standardizer mismatch on {f}"
assert ck_cfg["epochs"]==450 and ck_cfg["lr_decay"] and ck_cfg["ema"], f"not the frozen recipe: {ck_extra}"
print("frozen P_theta:", ck_extra)
t0=time.time(); A = sample_ptheta(model, sched, RUN.draws["A"].n, RUN.draws["A"].seed, RUN, DEVICE, verbose=False)
res["timings"]["gate_draw_s"]=time.time()-t0
gate = run_gate(A.z, ts.std, RUN, ref=reference_paths(RUN)); print_report(gate, columns=False)
res["stages"]["gate"]=summary_dict(gate); save()
assert gate.passed, "the frozen prior failed its gate on this device -- investigate before comparing samplers"

### Stage 2 — solve draw, dual at C3, and $h_\psi$

In [ ]:
q = q_params(); spec = CONSTRAINT_LEVELS[LEVEL]
t0=time.time(); POOL = sample_ptheta(model, sched, N_POOL, RUN.solve_draw.seed, RUN, DEVICE, verbose=False)
res["timings"]["pool_draw_s"]=time.time()-t0
pool_paths = ts.std.to_paths(POOL.z, "pool")
cs = build(pool_paths, spec["testfuns"], spec["vanillas"], q)
t0=time.time(); tilt, r, _ = solve_level(LEVEL, pool_paths, q); res["timings"]["dual_s"]=time.time()-t0
print(f"dual {LEVEL}: m={cs.m}  ESS {r.ess_frac*100:.2f}%  KL {r.kl:.5f}  |beta_raw| {np.linalg.norm(r.beta_raw):.2f}")
res["stages"]["dual"]=dict(m=int(cs.m), ess=float(r.ess_frac), kl=float(r.kl)); save()

B = sample_ptheta(model, sched, RUN.draws["B"].n, RUN.draws["B"].seed, RUN, DEVICE, verbose=False)
from taskc.run_hpsi import targets_for
LB = targets_for(tilt, ts.std, B.z, q)
hnet = HNet(RUN.data_dim, 256, 32, 0.05)
t0=time.time(); hlog = train_hnet(hnet, B.z, LB, sched, device=DEVICE, epochs=100, verbose=False)
res["timings"]["hpsi_s"]=time.time()-t0
save_hnet(os.path.join(DRIVE,"hpsi_C3.pt"), hnet, dict(level=LEVEL, E_L_B=float(LB.mean())))
print(f"h_psi: {res['timings']['hpsi_s']/60:.1f} min  L* on B in [{LB.min():.3f}, {LB.max():.3f}]  MSE {hlog.epoch_loss[-1]:.2e}")
res["stages"]["hpsi"]=dict(E_L_B=float(LB.mean()), L_min=float(LB.min()), L_max=float(LB.max()), mse=hlog.epoch_loss); save()

### Stage 3 — comparison A

Both arms produce $N$ **unweighted** paths. SIR's cost is one uncorrected draw of $N$ plus resampling;
the sampler's is one corrected draw of $N$. Standard errors are put on a common footing by
$\mathrm{SE}\sqrt{t}$, which is invariant to the budget because $\mathrm{SE}\propto 1/\sqrt{N}$ and
$t \propto N$; the ratio of these is the matched-compute ratio. $h_\psi$'s training is reported both
ways: excluded (steady state, the sampler reused) and amortized over this single use.

In [ ]:
def price_all(z):
    p = ts.std.to_paths(z)
    return {k:(float(v), float(s)) for k,(v,s) in ev.price_exotics(p).items()}

# arm A: corrected sampler
t0=time.time(); SMT = sample_smt(model, hnet, sched, N_EVAL, 7001, RUN, DEVICE, verbose=False)
t_smt = time.time()-t0; px_smt = price_all(SMT.z)

# arm S: SIR -- fresh pool of N, closed-form L*, resample with replacement
t0=time.time(); FRESH = sample_ptheta(model, sched, N_EVAL, 7002, RUN, DEVICE, verbose=False)
t_pool = time.time()-t0
fp = ts.std.to_paths(FRESH.z)
G = build(fp, spec["testfuns"], spec["vanillas"], q).G
logL = tilt.logL(G); w = np.exp(logL - logL.max()); w /= w.sum()
ess_fresh = float(1.0/np.sum(w**2)/len(w))
t0=time.time(); idx = np.random.default_rng(7003).choice(len(w), size=N_EVAL, replace=True, p=w)
t_res = time.time()-t0
px_sir = price_all(FRESH.z[idx])
uniq = int(len(np.unique(idx)))
# empirical SIR standard error over R independent resamples of the same pool
rng = np.random.default_rng(7004); reps={k:[] for k in EXOTICS}
for i in range(R_SIR):
    j = rng.choice(len(w), size=N_EVAL, replace=True, p=w)
    for k,v in price_all(FRESH.z[j]).items(): reps[k].append(v[0])
sir_sd = {k: float(np.std(v, ddof=1)) for k,v in reps.items()}
# weighted reference (weights kept, not resampled)
wref = {k: (float(np.average(ev.exotic_payoff(fp, **sp), weights=w)),
            float(np.sqrt(np.sum(w**2*(ev.exotic_payoff(fp, **sp)-np.average(ev.exotic_payoff(fp, **sp), weights=w))**2))))
        for k,sp in EXOTICS.items()}
t_sir = t_pool + t_res
print(f"ESS on the fresh pool {ess_fresh*100:.2f}%   unique draws retained {uniq:,}/{N_EVAL:,} = {uniq/N_EVAL*100:.1f}%")
print(f"time: sampler {t_smt:.1f}s   SIR {t_sir:.1f}s (pool {t_pool:.1f} + resample {t_res:.2f})   premium {t_smt/t_pool:.2f}x")
print(f"\n{'exotic':28s} {'sampler':>18s} {'SIR':>18s} {'weighted':>18s} {'SE*sqrt(t) smt':>15s} {'SIR':>10s} {'ratio':>7s}")
rows={}
for k in EXOTICS:
    a,sa = px_smt[k]; b,_ = px_sir[k]; sb = sir_sd[k]; wv,ws = wref[k]
    ea, eb = sa*math.sqrt(t_smt/60), sb*math.sqrt(t_sir/60)
    rows[k]=dict(smt=[a,sa], sir=[b,sb], weighted=[wv,ws], eff_smt=ea, eff_sir=eb, ratio=eb/ea)
    print(f"  {k:26s} {a:8.4f}+-{sa:.4f} {b:8.4f}+-{sb:.4f} {wv:8.4f}+-{ws:.4f} {ea:15.5f} {eb:10.5f} {eb/ea:7.2f}")
res["stages"]["A"]=dict(t_smt=t_smt, t_sir=t_sir, t_pool=t_pool, t_hpsi=res["timings"]["hpsi_s"],
                        ess_fresh=ess_fresh, unique=uniq, unique_frac=uniq/N_EVAL, premium=t_smt/t_pool, exotics=rows); save()
print("\nratio > 1 means the sampler is more efficient at matched compute.")
print(f"amortized over this single use, the sampler costs {(t_smt+res['timings']['hpsi_s'])/t_sir:.2f}x the SIR wall-clock.")

### Stage 4 — crossover sweep (mechanism test, payoff-directed tilts)

For a ladder of $\gamma$ the measure is $w \propto \exp(\beta^\top g + \gamma f)$ with $\beta$ re-solved so
the 53 equalities still hold. **These are not pricing configurations** — the tilt points at the payoff —
but they give a controlled ESS ladder from 93 % down to about 32 %, which no legitimate constraint set
in this project reaches. At each rung: SIR's empirical standard error from the weights, and the
sampler's from an $h_\psi$ trained on that rung's targets with the frozen recipe.

In [ ]:
f_sweep = np.asarray(ev.exotic_payoff(pool_paths, **EXOTICS[SWEEP_EXOTIC]), dtype=np.float64)
Gs, c_std, keep = standardise(np.asarray(cs.G, dtype=np.float64), cs.c.astype(np.float64))
beta0,_,_,_,_ = solve_beta(Gs, c_std, np.zeros(Gs.shape[0]), np.zeros(Gs.shape[1]))
# walk gamma down until each target ESS is first reached
ladder, beta, gamma, step = [], beta0.copy(), 0.0, -0.25/max(f_sweep.std(),1e-12)
want = list(TARGET_ESS)
for it in range(60):
    beta, wv, z, lse, rr = solve_beta(Gs, c_std, gamma*f_sweep, beta)
    e = float(1.0/np.sum(wv**2)/len(wv))
    if want and e <= want[0] + 0.005:
        ladder.append(dict(gamma=float(gamma), ess=e, kl=kl_of(wv,z,lse), beta=beta.copy(), w=wv.copy()))
        print(f"  rung {len(ladder)}: gamma {gamma:+.4f}  ESS {e*100:5.2f}%  KL {kl_of(wv,z,lse):.4f}")
        want.pop(0)
        if not want: break
    gamma += step
print(f"{len(ladder)} rungs")

sweep=[]
for i,L in enumerate(ladder):
    wv = L["w"]; pay = f_sweep
    ref = float(np.average(pay, weights=wv))
    # SIR: empirical SE over R resamples of N_EVAL from the 1e6 pool
    rng = np.random.default_rng(8000+i); vals=[]
    for _ in range(R_SIR):
        j = rng.choice(len(wv), size=N_EVAL, replace=True, p=wv); vals.append(float(pay[j].mean()))
    sir_se = float(np.std(vals, ddof=1)); uniq_frac = float(len(np.unique(j))/N_EVAL)
    # sampler: h_psi on this rung's targets, then a corrected draw
    zz = Gs @ L["beta"] + L["gamma"]*f_sweep            # log L* up to the normaliser
    Lz = np.exp(zz - (np.log(np.mean(np.exp(zz - zz.max()))) + zz.max()))
    n_h = min(RUN.draws["B"].n, len(Lz))
    sub = np.random.default_rng(9000+i).choice(len(Lz), size=n_h, replace=False)
    hn = HNet(RUN.data_dim, 256, 32, 0.05)
    t0=time.time(); train_hnet(hn, POOL.z[sub], Lz[sub], sched, device=DEVICE, epochs=100, verbose=False); t_h=time.time()-t0
    t0=time.time(); S = sample_smt(model, hn, sched, N_EVAL, 9500+i, RUN, DEVICE, verbose=False); t_s=time.time()-t0
    ps = ts.std.to_paths(S.z); v = ev.exotic_payoff(ps, **EXOTICS[SWEEP_EXOTIC])
    smt_px, smt_se = float(v.mean()), float(v.std(ddof=1)/math.sqrt(len(v)))
    eff_sir, eff_smt = sir_se*math.sqrt((t_pool)/60), smt_se*math.sqrt(t_s/60)
    row=dict(gamma=L["gamma"], ess=L["ess"], kl=L["kl"], weighted_ref=ref, sir_se=sir_se, unique_frac=uniq_frac,
             smt_price=smt_px, smt_se=smt_se, bias=smt_px-ref, t_hpsi=t_h, t_smt=t_s,
             eff_sir=eff_sir, eff_smt=eff_smt, ratio=eff_sir/eff_smt)
    sweep.append(row); res["stages"]["sweep"]=sweep; save()
    print(f"  ESS {L['ess']*100:5.2f}%  SIR SE {sir_se:.5f} (uniq {uniq_frac*100:4.1f}%)  sampler SE {smt_se:.5f}  "
          f"bias {smt_px-ref:+.4f}  eff ratio {eff_sir/eff_smt:5.2f}  ({t_h+t_s:.0f}s)", flush=True)
res["timings"]["total_s"]=float(sum(v for v in res["timings"].values())); save()
print("\nratio > 1 -> sampler more efficient. Crossover is where the ratio passes 1.")

### Stage 5 — figure

In [ ]:
sw = res["stages"]["sweep"]
fig, ax = plt.subplots(1,3, figsize=(13.5,4.2))
e = [s["ess"]*100 for s in sw]
ax[0].plot(e,[s["eff_sir"] for s in sw],"-o",ms=4,label="SIR"); ax[0].plot(e,[s["eff_smt"] for s in sw],"-s",ms=4,label="corrected sampler")
ax[0].set_xlabel("ESS (%)"); ax[0].set_ylabel(r"SE $\times\sqrt{t}$  (lower is better)"); ax[0].legend(fontsize=7); ax[0].grid(alpha=.25)
ax[1].plot(e,[s["ratio"] for s in sw],"-o",ms=4,color="C2"); ax[1].axhline(1,ls="--",c="k",lw=1)
ax[1].axvline(58.6,ls=":",c="0.5",lw=1); ax[1].set_xlabel("ESS (%)"); ax[1].set_ylabel("SIR / sampler efficiency")
ax[1].set_title("ratio > 1: sampler wins",fontsize=9); ax[1].grid(alpha=.25)
ax[2].plot(e,[s["unique_frac"]*100 for s in sw],"-o",ms=4,color="C4"); ax[2].set_xlabel("ESS (%)")
ax[2].set_ylabel("unique draws retained (%)"); ax[2].grid(alpha=.25)
for a in ax: a.invert_xaxis()
fig.tight_layout(); fig.savefig(os.path.join(DRIVE,"sir_crossover.pdf"),bbox_inches="tight"); plt.show()
print("saved", os.path.join(DRIVE,"sir_crossover.pdf"))

Results in `sir_vs_sampler.json`. The sweep is a mechanism test on payoff-directed tilts and must be labelled as such wherever it appears.